In [4]:
# DSA 210 - Phase 3: Machine Learning
# Ulas Alpandiner - 33831
#
# can we predict if a fund will beat the deposit rate next month?

import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# load data (same as phase 2)
tefas_file = glob.glob("TEFAS*.csv")[0]
evds_file  = glob.glob("EVDS*.xlsx")[0]

funds = pd.read_csv(tefas_file, parse_dates=["Tarih"])
funds.columns = ["Date", "Code", "Name", "Price", "Shares", "Investors", "TotalValue"]

rates = pd.read_excel(evds_file).iloc[:, :2]
rates.columns = ["RawDate", "DepositAPR"]
rates["Date"] = pd.to_datetime(rates["RawDate"], errors="coerce", dayfirst=True)
rates["DepositAPR"] = pd.to_numeric(rates["DepositAPR"], errors="coerce")
rates = rates.dropna(subset=["Date", "DepositAPR"])[["Date", "DepositAPR"]]

daily_idx = pd.date_range(rates["Date"].min(), funds["Date"].max(), freq="D")
rates = rates.set_index("Date").reindex(daily_idx).ffill().rename_axis("Date").reset_index()
rates["Rf"] = rates["DepositAPR"] / 100 / 365

# categorise funds
def categorise(name):
    n = str(name).upper()
    if "PARA PİYASASI" in n: return "Money Market"
    if "ALTIN" in n or "KIYMETLİ MADEN" in n: return "Gold"
    if "HİSSE" in n: return "Equity"
    if "EUROBOND" in n: return "Eurobond"
    if "BORÇLANMA" in n: return "Debt"
    if "FON SEPETİ" in n: return "Fund of Funds"
    if "KARMA" in n: return "Mixed"
    if "DEĞİŞKEN" in n: return "Variable"
    if "KATILIM" in n: return "Participation"
    if "SERBEST" in n: return "Hedge"
    return "Other"

funds["Category"] = funds["Name"].apply(categorise)
funds = funds.sort_values(["Code", "Date"])
funds["Return"] = funds.groupby("Code")["Price"].pct_change()
funds = funds[funds["Return"].abs() < 0.5]
funds = funds.merge(rates[["Date", "Rf", "DepositAPR"]], on="Date", how="left")
funds["Excess"] = funds["Return"] - funds["Rf"]
funds["YM"] = funds["Date"].dt.to_period("M")

print("Total observations:", len(funds))


# ---- build monthly features ----
# for each fund, for each month, summarise the daily data
monthly = funds.groupby(["Code", "Category", "YM"]).agg(
    MeanReturn    = ("Return", "mean"),
    Volatility    = ("Return", "std"),
    MeanExcess    = ("Excess", "mean"),
    TradingDays   = ("Return", "count"),
    DepositRate   = ("DepositAPR", "mean"),
    InvestorStart = ("Investors", "first"),
    InvestorEnd   = ("Investors", "last"),
    ValueStart    = ("TotalValue", "first"),
    ValueEnd      = ("TotalValue", "last"),
).reset_index()

monthly["InvestorGrowth"] = (monthly["InvestorEnd"] / monthly["InvestorStart"]) - 1
monthly["ValueGrowth"]    = (monthly["ValueEnd"] / monthly["ValueStart"]) - 1
monthly = monthly[monthly["TradingDays"] >= 10]

# target: did the fund beat the deposit NEXT month?
monthly = monthly.sort_values(["Code", "YM"])
monthly["NextExcess"] = monthly.groupby("Code")["MeanExcess"].shift(-1)
monthly = monthly.dropna(subset=["NextExcess"])
monthly["Target"] = (monthly["NextExcess"] > 0).astype(int)

# encode category
le = LabelEncoder()
monthly["Cat"] = le.fit_transform(monthly["Category"])

print("Monthly rows:", len(monthly))
print("Beat rate:", f"{monthly['Target'].mean():.1%}")


#prepare features
features = ["MeanReturn", "Volatility", "MeanExcess", "DepositRate",
            "InvestorGrowth", "ValueGrowth", "TradingDays", "Cat"]

ml = monthly[features + ["Target"]].replace([np.inf, -np.inf], np.nan).dropna()
X = ml[features]
y = ml["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")


#train 3 models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
}

results = []
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_sc, y_train)
        pred = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    results.append([name, acc])
    print(f"\n--- {name} (accuracy: {acc:.3f}) ---")
    print(classification_report(y_test, pred, target_names=["NotBeat", "Beat"]))

# baseline: always guess majority class
baseline = (y_test == y_test.mode()[0]).mean()
print(f"Baseline (majority class): {baseline:.3f}")


#save results
res_df = pd.DataFrame(results, columns=["Model", "Accuracy"]).sort_values("Accuracy", ascending=False)
res_df.to_csv("ml_results.csv", index=False)


#plot 1: model comparison
plt.figure(figsize=(8, 4))
bars = plt.bar(res_df["Model"], res_df["Accuracy"], color=["#2874A6", "#1ABC9C", "#E74C3C"])
plt.axhline(baseline, color="gray", ls="--", label=f"Baseline {baseline:.3f}")
for b, a in zip(bars, res_df["Accuracy"]):
    plt.text(b.get_x() + b.get_width()/2, a + 0.005, f"{a:.3f}", ha="center")
plt.ylabel("Accuracy"); plt.title("Model Comparison")
plt.legend(); plt.tight_layout()
plt.savefig("fig5_model_comparison.png"); plt.close()


#plot 2: confusion matrix (best model)
best = res_df.iloc[0]["Model"]
m = models[best]
pred_best = m.predict(X_test_sc) if best == "Logistic Regression" else m.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_best,
    display_labels=["NotBeat", "Beat"], cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix - {best}")
plt.tight_layout(); plt.savefig("fig6_confusion_matrix.png"); plt.close()


#plot 3: feature importance (random forest)
rf = models["Random Forest"]
imp = pd.DataFrame({"Feature": features, "Importance": rf.feature_importances_})
imp = imp.sort_values("Importance")

plt.figure(figsize=(8, 5))
plt.barh(imp["Feature"], imp["Importance"], color="#2874A6")
plt.xlabel("Importance"); plt.title("Feature Importance (Random Forest)")
plt.tight_layout(); plt.savefig("fig7_feature_importance.png"); plt.close()

imp.to_csv("feature_importance.csv", index=False)


Total observations: 1344884
Monthly rows: 62457
Beat rate: 56.8%
Train: 49960 | Test: 12490

--- Logistic Regression (accuracy: 0.661) ---
              precision    recall  f1-score   support

     NotBeat       0.61      0.60      0.60      5400
        Beat       0.70      0.71      0.70      7090

    accuracy                           0.66     12490
   macro avg       0.65      0.65      0.65     12490
weighted avg       0.66      0.66      0.66     12490


--- Decision Tree (accuracy: 0.730) ---
              precision    recall  f1-score   support

     NotBeat       0.68      0.72      0.70      5400
        Beat       0.78      0.74      0.76      7090

    accuracy                           0.73     12490
   macro avg       0.73      0.73      0.73     12490
weighted avg       0.73      0.73      0.73     12490


--- Random Forest (accuracy: 0.708) ---
              precision    recall  f1-score   support

     NotBeat       0.73      0.52      0.60      5400
        Beat    